# CS-323 Artificial Intelligence: Complex Engineering Problem (CEP)
## Term Project: Intelligent Vacuum-Cleaning Robot

**Group Members:**
1. **[Member 1 Name]** — `[CS-24XXX]`
2. **[Member 2 Name]** — `[CS-24XXX]`
3. **[Member 3 Name]** — `[CS-24XXX]`
4. **[Member 4 Name]** — `[CS-24XXX]`

---
### Project Overview
This notebook implements an **Intelligent Vacuum-Cleaning Robot** operating in a 2D grid environment. It integrates two classical Artificial Intelligence paradigms:
1. **Constraint Satisfaction Problem (CSP):** Determines a valid cleaning order across available time slots while satisfying precedence, latest-slot, and slot-exclusivity constraints using **Backtracking Search**, **MRV + Degree heuristics**, **LCV ordering**, and **Forward Checking**.
2. **Graph-Based Search:** Computes collision-free 4-directional paths between the charging station and dirty locations using **Breadth-First Search (BFS)** (uninformed) and **A\* Search** with Manhattan distance heuristic (informed).

## 0. Input Files Setup & Verification
The required `.txt` files (`Environment.txt`, `Schedule.txt`, `Environment_Sample.txt`, `Schedule_Sample.txt`, `Environment_Error.txt`, and `Schedule_Error.txt`) are loaded dynamically at runtime. The cell below verifies/creates these input files in the working directory so the notebook can be executed end-to-end without hardcoded parameters.

In [1]:
import collections
import heapq
import copy
import re
import os

# Create the 6 required input files if not already present in the working directory
input_files = {
    "Environment.txt": """10,10
12
(1,2) (1,3) (2,3) (3,3) (4,3) (4,6) (5,6) (6,6) (7,2) (7,3) (8,5) (8,6)
5
(0,9) (3,8) (8,8) (9,1) (5,2)
(0,0)""",
    "Schedule.txt": """5
4
D1,D3
D2,D4
D1,D5
D5,D4
3
D2,T3
D1,T2
D3,T4""",
    "Environment_Sample.txt": """8,8
6
(1,3) (2,3) (3,3) (4,3) (5,3) (6,3)
4
(0,7) (3,7) (7,7) (7,0)
(0,0)""",
    "Schedule_Sample.txt": """4
3
D1,D3
D2,D4
D1,D4
2
D3,T3
D2,T3""",
    "Environment_Error.txt": """8,8
8
(1,3) (2,3) (3,3) (4,3) (5,3) (6,3) (0,6) (1,7)
4
(0,7) (3,7) (7,7) (7,0)
(0,0)""",
    "Schedule_Error.txt": """5
3
D1,D2
D2,D3
D3,D1
2
D1,T3
D2,T4"""
}

for filename, content in input_files.items():
    if not os.path.exists(filename):
        with open(filename, "w") as f:
            f.write(content.strip() + "\n")


print("Input files verified in working directory:", [f for f in input_files if os.path.exists(f)])

Input files verified in working directory: ['Environment.txt', 'Schedule.txt', 'Environment_Sample.txt', 'Schedule_Sample.txt', 'Environment_Error.txt', 'Schedule_Error.txt']


## 1. Environment & Schedule Input Parsers
* `parse_environment(file_path)`: Reads grid dimensions (`rows,cols`), obstacle coordinates, dirty location coordinates (assigned labels `D1, D2, ...` in order of appearance), and the charging station coordinate.
* `parse_schedule(file_path, num_dirty)`: Reads available time slots (`T1, T2, ...`), precedence pairs `(D_before, D_after)`, and latest-slot rules `(D_task, T_slot)`. Both parsers tolerate whitespace (`(1, 2)`), accept pairs written as `D1,D3` or `(D1,D3)`, and validate the input (counts, bounds, overlaps, unknown task labels, slots >= tasks). `check_reachability` flags dirty cells that cannot be reached from the charging station.

In [2]:
# Matches a coordinate such as "(3,7)" or "(3, 7)" and captures the row and column numbers.
COORDINATE_PATTERN = re.compile(r'\(\s*(\d+)\s*,\s*(\d+)\s*\)')


def _read_coordinates(file_lines, line_number, how_many):
    """Read `how_many` (row,col) tuples starting at file_lines[line_number].
    The tuples may be spread over several lines. Returns (coordinates, next_line_number)."""
    coordinates = []
    while len(coordinates) < how_many:
        if line_number >= len(file_lines):
            raise ValueError(f"Expected {how_many} coordinates but found only {len(coordinates)}.")
        for row_text, col_text in COORDINATE_PATTERN.findall(file_lines[line_number]):
            coordinates.append((int(row_text), int(col_text)))
        line_number += 1
    if len(coordinates) != how_many:
        raise ValueError(f"Expected {how_many} coordinates but found {len(coordinates)}.")
    return coordinates, line_number


def parse_environment(file_path):
    """Load Environment.txt -> (grid, charging_station, dirty_locations). Raises ValueError if malformed."""
    with open(file_path, 'r', encoding='utf-8') as env_file:
        file_lines = [text.strip() for text in env_file if text.strip()]   # ignore blank lines
    if len(file_lines) < 3:
        raise ValueError("Environment file is incomplete.")

    # Line 1: grid size as "rows,columns"
    num_rows, num_cols = (int(number) for number in file_lines[0].split(','))
    line_number = 1

    # Obstacles: a count, then that many coordinates
    num_obstacles = int(file_lines[line_number]); line_number += 1
    if num_obstacles:
        obstacle_coordinates, line_number = _read_coordinates(file_lines, line_number, num_obstacles)
    else:
        obstacle_coordinates = []

    # Dirty locations: a count, then that many coordinates (first = D1, second = D2, ...)
    num_dirty = int(file_lines[line_number]); line_number += 1
    if num_dirty:
        dirty_locations, line_number = _read_coordinates(file_lines, line_number, num_dirty)
    else:
        dirty_locations = []

    # Charging station: exactly one coordinate
    (charging_station,), line_number = _read_coordinates(file_lines, line_number, 1)
    obstacle_set = set(obstacle_coordinates)   # a set makes "is this cell blocked?" checks fast

    # ---- Validate the input ----
    if len(obstacle_set) != num_obstacles:
        raise ValueError("Duplicate obstacle coordinates found.")
    if len(set(dirty_locations)) != num_dirty:
        raise ValueError("Duplicate dirty-location coordinates found.")
    cells_to_check = (("Obstacle", obstacle_coordinates),
                      ("Dirty location", dirty_locations),
                      ("Charging station", [charging_station]))
    for cell_kind, cell_list in cells_to_check:
        for (row, col) in cell_list:
            if not (0 <= row < num_rows and 0 <= col < num_cols):
                raise ValueError(f"{cell_kind} {(row, col)} lies outside the {num_rows}x{num_cols} grid.")
    if charging_station in obstacle_set:
        raise ValueError("Charging station overlaps an obstacle.")
    if charging_station in dirty_locations:
        raise ValueError("Charging station overlaps a dirty location.")
    dirty_on_obstacle = obstacle_set.intersection(dirty_locations)
    if dirty_on_obstacle:
        raise ValueError(f"Dirty location(s) overlap obstacles: {sorted(dirty_on_obstacle)}")

    # The keys 'rows', 'cols', 'obstacles' are used by the search code, so keep these names.
    grid = {'rows': num_rows, 'cols': num_cols, 'obstacles': obstacle_set}
    return grid, charging_station, dirty_locations


def parse_schedule(file_path, num_dirty):
    """Load Schedule.txt. Accepts pairs written as `D1,D3` or `(D1,D3)`. Raises ValueError if invalid."""
    with open(file_path, 'r', encoding='utf-8') as schedule_file:
        file_lines = [text.strip() for text in schedule_file if text.strip()]   # ignore blank lines

    def split_pair(line_text):
        """Turn a line like '(D1,D3)' or 'D1, D3' into ('D1', 'D3')."""
        pieces = re.sub(r'[()\s]', '', line_text).split(',')   # drop brackets and spaces
        if len(pieces) != 2 or not all(pieces):
            raise ValueError(f"Malformed constraint line: '{line_text}'")
        return pieces[0].upper(), pieces[1].upper()

    # Line 1: number of time slots
    num_slots = int(file_lines[0])
    if num_slots < num_dirty:
        raise ValueError(f"Number of time slots ({num_slots}) must be >= number of dirty locations ({num_dirty})")

    # Precedence constraints: (task_first, task_second) means task_first is cleaned before task_second
    num_precedence = int(file_lines[1])
    line_number = 2
    precedence_pairs = []
    for _ in range(num_precedence):
        precedence_pairs.append(split_pair(file_lines[line_number]))
        line_number += 1

    # Latest-slot constraints: (task, slot) means the task must be done in that slot or earlier
    num_latest_slot = int(file_lines[line_number]); line_number += 1
    latest_slot_pairs = []
    for _ in range(num_latest_slot):
        latest_slot_pairs.append(split_pair(file_lines[line_number]))
        line_number += 1

    # ---- Validate labels: every task must be one of D1..Dn, every slot must look like T1, T2, ... ----
    valid_task_labels = {f"D{number + 1}" for number in range(num_dirty)}
    for task_first, task_second in precedence_pairs:
        for task_label in (task_first, task_second):
            if task_label not in valid_task_labels:
                raise ValueError(f"Precedence constraint refers to unknown task '{task_label}' (valid: D1..D{num_dirty}).")
    for task_label, slot_label in latest_slot_pairs:
        if task_label not in valid_task_labels:
            raise ValueError(f"Latest-slot constraint refers to unknown task '{task_label}' (valid: D1..D{num_dirty}).")
        if not re.fullmatch(r'T\d+', slot_label):
            raise ValueError(f"Invalid slot label '{slot_label}' (expected T1, T2, ...).")

    # The keys 'precedence', 'latest_slots', 'num_slots' are used by the CSP code, so keep these names.
    return {'precedence': precedence_pairs, 'latest_slots': latest_slot_pairs, 'num_slots': num_slots}


def check_reachability(grid, charging_station, dirty_locations):
    """Flood-fill from the charging station; return the dirty locations that cannot be reached."""
    reachable_cells = {charging_station}
    cells_to_visit = [charging_station]
    while cells_to_visit:
        current_cell = cells_to_visit.pop()
        for neighbor_cell, _action, _step_cost in get_successors(current_cell, grid):
            if neighbor_cell not in reachable_cells:
                reachable_cells.add(neighbor_cell)
                cells_to_visit.append(neighbor_cell)
    return [(f"D{task_number + 1}", location)
            for task_number, location in enumerate(dirty_locations)
            if location not in reachable_cells]


## 2. Graph-Based Search Module (Robot Navigation)
Implements the required search functions for 4-directional grid navigation (`UP`, `DOWN`, `LEFT`, `RIGHT`) with uniform step cost `1`:
* `get_successors(state, grid)`: Returns valid adjacent `(next_state, action, step_cost)` tuples within grid boundaries and free of obstacles.
* `goal_test(state, goal_state)`: Checks if `state == goal_state`.
* `path_cost(path)`: Computes total movement cost (`len(path) - 1`, or `0` for empty/single-state paths).
* `heuristic(state, goal_state)`: Computes Manhattan distance $h(n) = |r_n - r_{goal}| + |c_n - c_{goal}|$, which is admissible and consistent for 4-connected grids.
* `search(grid, start, goal, algorithm="A*")`: Supports **BFS** (uninformed) and **A\*** (informed), returning `(path, cost, nodes_expanded)`.

In [3]:
def get_successors(state, grid):
    """Return every cell the robot can step to from `state`, as (next_state, action, step_cost)."""
    row, col = state
    successors = []
    # The four allowed moves: (row change, column change, action name). No diagonals.
    possible_moves = [(-1, 0, 'UP'), (1, 0, 'DOWN'), (0, -1, 'LEFT'), (0, 1, 'RIGHT')]

    for row_change, col_change, action in possible_moves:
        next_row, next_col = row + row_change, col + col_change
        inside_grid = 0 <= next_row < grid['rows'] and 0 <= next_col < grid['cols']
        if inside_grid and (next_row, next_col) not in grid['obstacles']:
            successors.append(((next_row, next_col), action, 1))   # every step costs 1
    return successors


def goal_test(state, goal_state):
    """True if the robot is standing on the goal cell."""
    return state == goal_state


def path_cost(path):
    """Total cost of a path = number of moves = number of cells - 1 (0 for an empty or 1-cell path)."""
    if not path or len(path) <= 1:
        return 0
    return len(path) - 1


def heuristic(state, goal_state):
    """Manhattan distance: the fewest moves needed if there were no obstacles."""
    return abs(state[0] - goal_state[0]) + abs(state[1] - goal_state[1])


def search(grid, start, goal, algorithm="A*"):
    """Find a path from `start` to `goal`. Returns (path, cost, nodes_expanded).
    If no path exists it returns (None, infinity, nodes_expanded)."""
    if start == goal:
        return [start], 0, 0

    nodes_expanded = 0   # how many cells we pulled off the frontier and explored

    if algorithm == "BFS":
        # Breadth-First Search: explore cells in rings of equal distance, using a first-in-first-out queue.
        frontier = collections.deque([(start, [start])])   # each entry: (cell, path taken to reach it)
        visited_cells = {start}

        while frontier:
            current_cell, path_so_far = frontier.popleft()
            nodes_expanded += 1

            for next_cell, action, step_cost in get_successors(current_cell, grid):
                if next_cell not in visited_cells:
                    extended_path = path_so_far + [next_cell]
                    if goal_test(next_cell, goal):
                        return extended_path, path_cost(extended_path), nodes_expanded
                    visited_cells.add(next_cell)
                    frontier.append((next_cell, extended_path))

    elif algorithm == "A*":
        # A* Search: always expand the cell with the lowest f = cost_so_far + estimated_cost_to_goal.
        # Each frontier entry: (f_score, cell, path taken to reach it, cost_so_far)
        start_estimate = heuristic(start, goal)
        frontier = [(start_estimate, start, [start], 0)]
        best_cost_to_cell = {}   # cheapest known cost to reach each expanded cell

        while frontier:
            f_score, current_cell, path_so_far, cost_so_far = heapq.heappop(frontier)

            # Skip this entry if we already reached the same cell at an equal or lower cost
            if current_cell in best_cost_to_cell and best_cost_to_cell[current_cell] <= cost_so_far:
                continue

            best_cost_to_cell[current_cell] = cost_so_far
            nodes_expanded += 1

            if goal_test(current_cell, goal):
                return path_so_far, path_cost(path_so_far), nodes_expanded

            for next_cell, action, step_cost in get_successors(current_cell, grid):
                new_cost = cost_so_far + step_cost
                if next_cell not in best_cost_to_cell or best_cost_to_cell[next_cell] > new_cost:
                    new_f_score = new_cost + heuristic(next_cell, goal)
                    heapq.heappush(frontier, (new_f_score, next_cell, path_so_far + [next_cell], new_cost))

    return None, float('inf'), nodes_expanded   # frontier ran out: no path exists


## 3. Constraint Satisfaction Problem (CSP) Module (Task Scheduling)
Formulates cleaning task scheduling as a CSP:
* **Variables:** `D1, D2, ..., Dn` corresponding to dirty locations in `Environment.txt`.
* **Domains:** Available time slots `{T1, T2, ..., Tm}` where $m \ge n$.
* **Constraints:**
  1. *Slot Exclusivity (All-Different):* At most one task per time slot.
  2. *Precedence:* `D_before` must be assigned an earlier slot than `D_after`.
  3. *Latest-Slot:* `D_task` must be assigned a slot $\le$ `T_slot`.
* **Required Functions:** `select_unassigned_variable` (MRV + Degree tie-breaker), `order_domain_values` (LCV), `is_consistent`, `forward_check` (with domain restoration on backtrack), and `backtracking_search`.

In [4]:
def select_unassigned_variable(assignment, domains, constraints):
    """Choose which task to assign next: MRV first, then the degree heuristic to break ties."""
    unassigned_tasks = [task for task in domains.keys() if task not in assignment]

    # MRV (Minimum Remaining Values): find the task(s) with the fewest slots still possible
    tied_tasks = []
    fewest_slots_left = float('inf')
    for task in unassigned_tasks:
        slots_left = len(domains[task])
        if slots_left < fewest_slots_left:
            fewest_slots_left = slots_left
            tied_tasks = [task]            # new record low: start a fresh list
        elif slots_left == fewest_slots_left:
            tied_tasks.append(task)        # ties the record: keep it as well

    if len(tied_tasks) == 1:
        return tied_tasks[0]

    # Degree heuristic (tie-break): pick the task linked to the most other unassigned tasks
    highest_degree = -1
    best_task = tied_tasks[0]

    for task in tied_tasks:
        degree = 0
        for task_first, task_second in constraints['precedence']:
            # A precedence rule counts only if the task at its other end is still unassigned
            if (task_first == task and task_second not in assignment) or \
               (task_second == task and task_first not in assignment):
                degree += 1
        # Slot exclusivity links this task to every other unassigned task
        degree += len(unassigned_tasks) - 1

        if degree > highest_degree:
            highest_degree = degree
            best_task = task

    return best_task


def order_domain_values(variable, assignment, domains, constraints):
    """LCV (Least Constraining Value): order the slots so the one that removes the
    fewest options from the other unassigned tasks is tried first."""
    candidate_slots = domains[variable]
    if len(candidate_slots) <= 1:
        return candidate_slots

    other_unassigned_tasks = [task for task in domains.keys() if task not in assignment and task != variable]

    def count_eliminated(slot):
        """How many options for the other tasks would be ruled out if `variable` took `slot`?"""
        eliminated = 0
        slot_number = int(slot[1:])                       # 'T3' -> 3
        for other_task in other_unassigned_tasks:
            for other_slot in domains[other_task]:
                if other_slot == slot:                    # exclusivity: that slot would be taken
                    eliminated += 1
                    continue
                other_slot_number = int(other_slot[1:])
                for task_first, task_second in constraints['precedence']:
                    if variable == task_first and other_task == task_second:
                        if not (slot_number < other_slot_number):
                            eliminated += 1               # other_task would no longer be after variable
                    elif variable == task_second and other_task == task_first:
                        if not (other_slot_number < slot_number):
                            eliminated += 1               # other_task would no longer be before variable
        return eliminated

    return sorted(candidate_slots, key=count_eliminated)   # fewest eliminated first


def is_consistent(variable, value, assignment, constraints):
    """Can `variable` take slot `value` without breaking any constraint that can be checked right now?"""
    # 1. Slot exclusivity: no other task may already use this slot
    if value in assignment.values():
        return False

    slot_number = int(value[1:])

    # 2. Latest-slot constraint: the slot must not be later than the task's limit
    for task, latest_slot in constraints['latest_slots']:
        if task == variable:
            if slot_number > int(latest_slot[1:]):
                return False

    # 3. Precedence constraints: only checked against tasks that are already assigned
    for task_first, task_second in constraints['precedence']:
        if variable == task_first and task_second in assignment:
            second_slot_number = int(assignment[task_second][1:])
            if not (slot_number < second_slot_number):
                return False
        elif variable == task_second and task_first in assignment:
            first_slot_number = int(assignment[task_first][1:])
            if not (first_slot_number < slot_number):
                return False

    return True


def forward_check(variable, value, assignment, domains, constraints):
    """After assigning `value` to `variable`, remove slots that are no longer possible for the
    unassigned tasks. Returns the updated domains, or None if some task has no slot left."""
    updated_domains = copy.deepcopy(domains)     # work on a copy so backtracking can simply discard it
    assigned_slot_number = int(value[1:])

    for other_task in domains.keys():
        if other_task in assignment:             # already decided: nothing to prune
            continue

        allowed_slots = []
        for other_slot in updated_domains[other_task]:
            other_slot_number = int(other_slot[1:])

            # Rule 1: slot exclusivity - the slot just used is no longer available
            if other_slot == value:
                continue

            # Rule 2: precedence - drop slots that would break an ordering rule with `variable`
            slot_allowed = True
            for task_first, task_second in constraints['precedence']:
                if variable == task_first and other_task == task_second:
                    if not (assigned_slot_number < other_slot_number):
                        slot_allowed = False
                        break
                elif variable == task_second and other_task == task_first:
                    if not (other_slot_number < assigned_slot_number):
                        slot_allowed = False
                        break

            if slot_allowed:
                allowed_slots.append(other_slot)

        if not allowed_slots:                    # empty domain: this choice is a dead end
            return None
        updated_domains[other_task] = allowed_slots

    return updated_domains


def prune_latest_slots(domains, constraints):
    """Node consistency: remove slots later than a task's latest-slot constraint (unary constraints)."""
    pruned_domains = {task: list(slots) for task, slots in domains.items()}
    for task, latest_slot in constraints['latest_slots']:
        latest_slot_number = int(latest_slot[1:])
        pruned_domains[task] = [slot for slot in pruned_domains[task] if int(slot[1:]) <= latest_slot_number]
    return pruned_domains


def backtracking_search(variables, domains, constraints):
    """Find a complete task -> slot assignment, or return None if no feasible schedule exists."""

    def backtrack(assignment, current_domains):
        if len(assignment) == len(variables):         # every task has a slot: done
            return assignment

        next_task = select_unassigned_variable(assignment, current_domains, constraints)   # MRV + degree
        for slot in order_domain_values(next_task, assignment, current_domains, constraints):   # LCV order
            if is_consistent(next_task, slot, assignment, constraints):
                assignment[next_task] = slot

                # Forward checking works on a copy, so the old domains are restored automatically on backtrack
                domains_after_check = forward_check(next_task, slot, assignment, current_domains, constraints)
                if domains_after_check is not None:
                    complete_assignment = backtrack(assignment, domains_after_check)
                    if complete_assignment is not None:
                        return complete_assignment

                del assignment[next_task]             # dead end: undo this choice and try the next slot
        return None

    domains = prune_latest_slots(domains, constraints)
    if any(len(slots) == 0 for slots in domains.values()):
        return None                                   # a task has no feasible slot at all
    return backtrack({}, domains)


## 4. System Integration Module
`execute_schedule(grid, charging_station, schedule, algorithm="A*")` links the CSP output with the Graph-Based Search solver:
1. Initializes the robot at `charging_station`.
2. Iteratively navigates to each dirty location in the chronological order produced by the CSP solver, marking each location as cleaned upon arrival.
3. Computes a valid path back to `charging_station` once all dirty cells are cleaned.
4. Concatenates path segments without duplicating shared endpoints and reports failure immediately if any leg is unreachable.

In [5]:
def execute_schedule(grid, charging_station, schedule, algorithm="A*"):
    """
    Executes the CSP-generated schedule using graph-based search.
    schedule: list of (task_label, coordinate) ordered by time slot.
    Returns: (complete_path, total_cost, success)
    Side outputs: execute_schedule.last_legs (every leg attempted, including a failed one)
                  execute_schedule.cleaned (dirty cells reached, in order).
    """
    robot_position = charging_station          # the robot starts at the charging station
    complete_path = [charging_station]         # every cell the robot visits, in order
    total_cost = 0
    journey_legs = []                          # one record per trip between two locations
    cleaned_locations = []                     # dirty cells the robot has reached so far
    execute_schedule.last_legs = journey_legs
    execute_schedule.cleaned = cleaned_locations

    # The trips to make: each dirty cell in the CSP's order, then the way back to the charging station
    destinations = [(f"{task_label} {location}", location) for task_label, location in schedule]
    destinations.append((f"Charging Station {charging_station}", charging_station))
    print(destinations)
    previous_label = "Charging Station"        # name of the place the current trip starts from

    for stop_number, (destination_label, destination_position) in enumerate(destinations):
        # Use graph search to find a path from where the robot is to the next destination
        leg_path, leg_cost, leg_nodes_expanded = search(grid, robot_position, destination_position, algorithm)

        if leg_path is None:
            # No route exists: record the failed trip and report failure
            # instead of claiming the cleaning task was completed
            journey_legs.append({'from': previous_label, 'to': destination_label,
                                 'cost': None, 'nodes': leg_nodes_expanded, 'path': None})
            return complete_path, float('inf'), False

        journey_legs.append({'from': previous_label, 'to': destination_label,
                             'cost': leg_cost, 'nodes': leg_nodes_expanded, 'path': leg_path})
        complete_path.extend(leg_path[1:])     # skip the first cell: it is where the robot already is
        total_cost += leg_cost
        robot_position = destination_position

        is_dirty_stop = stop_number < len(schedule)    # the last stop is the charging station
        if is_dirty_stop:
            cleaned_locations.append(destination_position)   # robot arrived: mark as cleaned
        previous_label = destination_label

    return complete_path, total_cost, True


### Standalone search tests (edge cases)
Before integrating, the search component is tested on its own: BFS vs. A* on the same start/goal pair, the `start == goal` case, an unreachable goal, and a path-validity check (adjacent cells, no obstacles).

In [6]:
def is_valid_path(grid, path, start, goal):
    """A path is valid if it starts/ends correctly, steps between 4-neighbours and avoids obstacles/out-of-grid."""
    if not path or path[0] != start or path[-1] != goal:
        return False
    for (r1, c1), (r2, c2) in zip(path, path[1:]):
        if abs(r1 - r2) + abs(c1 - c2) != 1:
            return False
    return all(0 <= r < grid['rows'] and 0 <= c < grid['cols'] and (r, c) not in grid['obstacles'] for r, c in path)


sample_grid, sample_station, sample_dirty = parse_environment("Environment_Sample.txt")
print("1) Same start/goal solved by both algorithms:", sample_station, "->", sample_dirty[3])
for algo in ["BFS", "A*"]:
    p, c, n = search(sample_grid, sample_station, sample_dirty[3], algo)
    print(f"   {algo:<3} cost={c} nodes_expanded={n} valid_path={is_valid_path(sample_grid, p, sample_station, sample_dirty[3])}")
    print(f"       path={p}")

print("\n2) start == goal:")
for algo in ["BFS", "A*"]:
    print(f"   {algo:<3} ->", search(sample_grid, sample_station, sample_station, algo))

err_grid, err_station, err_dirty = parse_environment("Environment_Error.txt")
print("\n3) Unreachable goal (Environment_Error.txt):", err_station, "->", err_dirty[0])
for algo in ["BFS", "A*"]:
    print(f"   {algo:<3} ->", search(err_grid, err_station, err_dirty[0], algo), "(path, cost, nodes_expanded)")
print("   Reachability check flags:", check_reachability(err_grid, err_station, err_dirty))

print("\n4) path_cost edge cases: empty =", path_cost([]), "| single state =", path_cost([(0, 0)]), "| 3 states =", path_cost([(0, 0), (0, 1), (0, 2)]))

1) Same start/goal solved by both algorithms: (0, 0) -> (7, 0)
   BFS cost=7 nodes_expanded=20 valid_path=True
       path=[(0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (5, 0), (6, 0), (7, 0)]
   A*  cost=7 nodes_expanded=8 valid_path=True
       path=[(0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (5, 0), (6, 0), (7, 0)]

2) start == goal:
   BFS -> ([(0, 0)], 0, 0)
   A*  -> ([(0, 0)], 0, 0)

3) Unreachable goal (Environment_Error.txt): (0, 0) -> (0, 7)
   BFS -> (None, inf, 55) (path, cost, nodes_expanded)
   A*  -> (None, inf, 55) (path, cost, nodes_expanded)
   Reachability check flags: [('D1', (0, 7))]

4) path_cost edge cases: empty = 0 | single state = 0 | 3 states = 2


## 5. Visualization, Testing & Comparative Analysis
Runs all required test scenarios and compares **BFS** vs. **A\*** across total path cost and total nodes expanded:
1. **Test Run 1:** Custom Environment (`Environment.txt` & `Schedule.txt` — $10\times10$ grid, 12 obstacles, 5 dirty locations, 4 precedence constraints, 3 latest-slot constraints).
2. **Test Run 2:** Provided Sample Environment (`Environment_Sample.txt` & `Schedule_Sample.txt` — $8\times8$ grid, 6 obstacles, 4 dirty locations).
3. **Test Run 3A:** CSP Failure Case (`Environment.txt` & `Schedule_Error.txt` — cyclic precedence constraints).
4. **Test Run 3B:** Navigation Failure Case (`Environment_Error.txt` & `Schedule_Sample.txt` — unreachable dirty location `D1=(0,7)` walled off by obstacles).

In [7]:
def print_grid(grid, charging_station, dirty_locations, path=None, cleaned=None, title="Grid"):
    """C = charging station, ### = obstacle, D1.. = dirty, d1.. = cleaned, * = robot path, . = free."""
    print(f"--- {title} ---")
    path_set = set(path) if path else set()
    cleaned = set(cleaned) if cleaned else set()

    for r in range(grid['rows']):
        cells_out = []
        for c in range(grid['cols']):
            pos = (r, c)
            if pos == charging_station:
                s = "C"
            elif pos in grid['obstacles']:
                s = "###"
            elif pos in dirty_locations:
                idx = dirty_locations.index(pos) + 1
                s = f"d{idx}" if pos in cleaned else f"D{idx}"
            elif pos in path_set:
                s = "*"
            else:
                s = "."
            cells_out.append(f"{s:^4}")
        print("".join(cells_out))
    print("-" * (grid['cols'] * 4))


def verify_schedule(assignment, constraints, verbose=False):
    """Check every constraint on a complete assignment; with verbose=True print each check."""
    ok = True
    slots = list(assignment.values())
    exclusive = len(set(slots)) == len(slots)
    ok &= exclusive
    if verbose:
        print(f"  [{'OK' if exclusive else 'FAIL'}] Slot exclusivity: all {len(slots)} tasks use different slots")
    for before, after in constraints['precedence']:
        good = int(assignment[before][1:]) < int(assignment[after][1:])
        ok &= good
        if verbose:
            print(f"  [{'OK' if good else 'FAIL'}] {before} before {after}: {assignment[before]} < {assignment[after]}")
    for task, slot in constraints['latest_slots']:
        good = int(assignment[task][1:]) <= int(slot[1:])
        ok &= good
        if verbose:
            print(f"  [{'OK' if good else 'FAIL'}] {task} no later than {slot}: assigned {assignment[task]}")
    return ok


def run_test_case(env_file, sch_file, label):
    print("=" * 72)
    print(f"TEST CASE: {label}")
    print(f"Files: {env_file} | {sch_file}")
    print("=" * 72)

    try:
        grid, charging_station, dirty_locations = parse_environment(env_file)
        num_dirty = len(dirty_locations)
        constraints = parse_schedule(sch_file, num_dirty)
    except Exception as e:
        print(f"Initialization Error: {e}\n")
        return

    # 1. CSP Scheduling
    variables = [f"D{i+1}" for i in range(num_dirty)]
    domains = {v: [f"T{j+1}" for j in range(constraints['num_slots'])] for v in variables}

    print(f"Grid Size: {grid['rows']}x{grid['cols']} | Charging Station: {charging_station} | Dirty Spots: {num_dirty}")
    print(f"CSP Variables: {variables}")
    print(f"Precedence Constraints: {constraints['precedence']}")
    print(f"Latest-Slot Constraints: {constraints['latest_slots']}")
    unreachable = check_reachability(grid, charging_station, dirty_locations)
    if unreachable:
        print(f"WARNING: unreachable from the charging station: {unreachable}")

    assignment = backtracking_search(variables, domains, constraints)

    if assignment is None:
        print("STATUS: [CSP FAILURE] No feasible schedule exists satisfying all constraints.\n")
        return

    sorted_assignment = sorted(assignment.items(), key=lambda x: int(x[1][1:]))
    print("\nGenerated Schedule (time slot -> task):")
    for v, t in sorted_assignment:
        print(f"  {t}: {v} at {dirty_locations[int(v[1:]) - 1]}")
    print("Constraint verification:")
    is_valid = verify_schedule(assignment, constraints, verbose=True)
    print(f"All constraints satisfied: {is_valid}")

    task_map = {f"D{i+1}": dirty_locations[i] for i in range(num_dirty)}
    schedule_coords = [(v, task_map[v]) for v, t in sorted_assignment]
    print(f"Cleaning Visit Order: {[f'{v}:{coord}' for v, coord in schedule_coords]}")

    # 2. Integrated Navigation (BFS vs A*)
    results = {}
    for algo in ["BFS", "A*"]:
        print(f"\n--- Executing Integrated Simulation with {algo} ---")
        complete_path, total_cost, success = execute_schedule(grid, charging_station, schedule_coords, algo)
        legs = execute_schedule.last_legs
        cleaned = list(execute_schedule.cleaned)
        total_nodes = sum(l['nodes'] for l in legs)   # includes the failed leg's search effort

        for i, leg in enumerate(legs):
            if leg['path'] is None:
                print(f"  Leg {i+1}: {leg['from']} -> {leg['to']} | NO PATH FOUND | Nodes Expanded: {leg['nodes']}")
            else:
                print(f"  Leg {i+1}: {leg['from']} -> {leg['to']} | Cost: {leg['cost']} | Nodes Expanded: {leg['nodes']}")
                print(f"         Path: {leg['path']}")

        if not success:
            print(f"STATUS: [NAVIGATION FAILURE] Unreachable target encountered using {algo}; "
                  f"cleaned {len(cleaned)}/{num_dirty} locations. Task NOT completed.")
            results[algo] = {'cost': 'N/A', 'nodes': total_nodes, 'success': False}
            continue

        print(f"STATUS: [SUCCESS] All {num_dirty} locations cleaned & returned to Charging Station.")
        print(f"Complete journey ({len(complete_path)} positions): {complete_path}")
        print(f"Total Movements: {len(complete_path) - 1} | Total Navigation Cost: {total_cost} | Total Nodes Expanded: {total_nodes}")
        results[algo] = {'cost': total_cost, 'nodes': total_nodes, 'success': True, 'path': complete_path, 'cleaned': cleaned}

    # 3. Grid Visualization & Comparative Table
    if results["A*"]['success']:
        print()
        print_grid(grid, charging_station, dirty_locations, title="Initial Environment Grid")
        print_grid(grid, charging_station, dirty_locations, path=results["A*"]['path'],
                   cleaned=results["A*"]['cleaned'], title="Final Cleaned Grid & Complete Journey (A*)")

    bfs, ast = results['BFS'], results['A*']
    if bfs['success'] and ast['success']:
        # BFS is optimal on unit-cost grids, so BFS's cost is the optimal baseline
        optimal_row = ('baseline', 'Yes' if ast['cost'] == bfs['cost'] else 'No')
    else:
        optimal_row = ('N/A', 'N/A')
    print("\nComparative Analysis (BFS vs. A*):")
    print(f"{'Metric':<24} | {'BFS':<15} | {'A*':<15}")
    print("-" * 60)
    print(f"{'Execution Status':<24} | {str(bfs['success']):<15} | {str(ast['success']):<15}")
    print(f"{'Total Navigation Cost':<24} | {str(bfs['cost']):<15} | {str(ast['cost']):<15}")
    print(f"{'Total Nodes Expanded':<24} | {str(bfs['nodes']):<15} | {str(ast['nodes']):<15}")
    print(f"{'Cost optimal (vs BFS)':<24} | {optimal_row[0]:<15} | {optimal_row[1]:<15}")
    print("\n")


# Execute all required test scenarios
run_test_case("Environment.txt", "Schedule.txt", "1. Custom Environment & Schedule")
run_test_case("Environment_Sample.txt", "Schedule_Sample.txt", "2. Sample Environment & Schedule")
run_test_case("Environment.txt", "Schedule_Error.txt", "3A. Failure Case — Infeasible CSP Schedule")
run_test_case("Environment_Error.txt", "Schedule_Sample.txt", "3B. Failure Case — Unreachable Dirty Location")


TEST CASE: 1. Custom Environment & Schedule
Files: Environment.txt | Schedule.txt
Grid Size: 10x10 | Charging Station: (0, 0) | Dirty Spots: 5
CSP Variables: ['D1', 'D2', 'D3', 'D4', 'D5']
Precedence Constraints: [('D1', 'D3'), ('D2', 'D4'), ('D1', 'D5'), ('D5', 'D4')]
Latest-Slot Constraints: [('D2', 'T3'), ('D1', 'T2'), ('D3', 'T4')]

Generated Schedule (time slot -> task):
  T1: D1 at (0, 9)
  T2: D2 at (3, 8)
  T3: D3 at (8, 8)
  T4: D5 at (5, 2)
  T5: D4 at (9, 1)
Constraint verification:
  [OK] Slot exclusivity: all 5 tasks use different slots
  [OK] D1 before D3: T1 < T3
  [OK] D2 before D4: T2 < T5
  [OK] D1 before D5: T1 < T4
  [OK] D5 before D4: T4 < T5
  [OK] D2 no later than T3: assigned T2
  [OK] D1 no later than T2: assigned T1
  [OK] D3 no later than T4: assigned T3
All constraints satisfied: True
Cleaning Visit Order: ['D1:(0, 9)', 'D2:(3, 8)', 'D3:(8, 8)', 'D5:(5, 2)', 'D4:(9, 1)']

--- Executing Integrated Simulation with BFS ---
[('D1 (0, 9)', (0, 9)), ('D2 (3, 8)', 

## 6. Algorithm Design & Theoretical Analysis

### 1. Heuristic Justification (Manhattan Distance)
For 4-directional grid navigation (`UP`, `DOWN`, `LEFT`, `RIGHT`) with unit step cost $c = 1$:
$$h(n) = |r_n - r_{\text{goal}}| + |c_n - c_{\text{goal}}|$$
* **Admissibility:** Without obstacles, the minimum number of orthogonal steps between $(r_n, c_n)$ and $(r_{\text{goal}}, c_{\text{goal}})$ is exactly the Manhattan distance. Obstacles can only force detours that increase the true cost $h^*(n)$, so $h(n) \le h^*(n)$ always holds.
* **Consistency:** Moving one cell changes $|r_n - r_{\text{goal}}|$ or $|c_n - c_{\text{goal}}|$ by at most $1$, so $h(n) \le 1 + h(n')$, guaranteeing optimal paths without re-expanding closed nodes.

### 2. Graph Search Comparison (BFS vs. A\*)
* **BFS** explores frontiers uniformly in concentric layers ($O(b^d)$), guaranteeing shortest paths on unweighted grids but expanding many unnecessary nodes in directions away from the goal.
* **A\*** evaluates $f(n) = g(n) + h(n)$, prioritizing states closer to the target. Both algorithms find identical optimal path costs, while **A\*** expands substantially fewer nodes.

### 3. CSP Formulation & Heuristics
* **MRV + Degree Heuristic:** Selects the most constrained variable first (fewest legal time slots) and breaks ties using the variable involved in the highest number of active constraints with unassigned tasks, triggering fail-fast pruning.
* **LCV (Least Constraining Value):** Orders candidate slots to eliminate the fewest options from neighboring unassigned tasks.
* **Forward Checking:** Immediately prunes inconsistent time slots from unassigned neighbors after each assignment and restores domains cleanly upon backtracking.
* **Node-count note:** BFS applies the goal test when a node is *generated* while A\* applies it when a node is *expanded*, so the counts are slightly favourable to BFS; the gap in A\*'s favour is therefore conservative.
